# OmniCopilot — Cooperation Coverage Analysis

**First quantitative result (pre-Kill-Gate-A).**

Measures how many more objects the fleet sees *collectively* versus any *single agent* alone, using ground-truth objects. This is the **ceiling** of cooperative benefit available in the data (no detection model yet — that comes in Phase 2-3).

**How to use on Kaggle:**
1. Right sidebar -> Settings -> Internet -> **On**
2. Add your OPV2V dataset as Input (already mounted at `/kaggle/input/...`)
3. Run cells top to bottom.

Code comes from GitHub (`git clone`), data is the Kaggle dataset, they meet here.

## 1. Setup — clone the repo from GitHub and put it on the path

In [ ]:
import os
import sys
from pathlib import Path

# --- EDIT THESE ---
USER = "ShreerakshaM"          # your GitHub username / org
REPO = "OmniCopilot-ai"        # your repo name
PRIVATE = False                 # repo is public
# ------------------

clone_dir = f"/kaggle/working/{REPO}"

if not os.path.exists(clone_dir):
    if PRIVATE:
        # Store your GitHub token as a Kaggle Secret named 'GITHUB_TOKEN'
        # (Add-ons -> Secrets). Keeps the token OUT of the notebook.
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        os.system(f"git clone https://{token}@github.com/{USER}/{REPO}.git {clone_dir}")
    else:
        os.system(f"git clone https://github.com/{USER}/{REPO}.git {clone_dir}")
else:
    # Already cloned this session -> pull the latest.
    os.system(f"cd {clone_dir} && git pull")

sys.path.insert(0, f"{clone_dir}/python")
sys.path.insert(0, f"{clone_dir}/scripts")
print("Repo ready at:", clone_dir)

## 2. Verify imports and locate the dataset

In [ ]:
from omnicopilot.data.opv2v import OPV2VDataset
import analyze_cooperation_coverage as ac
print("Code loaded OK")

# Auto-find the OPV2V 'test' folder under /kaggle/input (robust to the nested path).
candidates = list(Path("/kaggle/input").rglob("test"))
candidates = [c for c in candidates if c.is_dir() and any(p.is_dir() for p in c.iterdir())]
print("Candidate data roots:")
for c in candidates:
    print("  ", c)

# Pick the first that contains scenario folders; EDIT if needed.
DATA_ROOT = candidates[0] if candidates else None
print("\nUsing DATA_ROOT =", DATA_ROOT)

## 3. Sanity check — one scenario, one frame

Confirms the loader reads real data and the cooperation signal is present
(agents seeing different subsets of objects).

In [ ]:
ds = OPV2VDataset(DATA_ROOT)
ds.load()
print("Scenarios found:", len(ds))

sid = ds.scenario_ids()[0]
scenario = ds.get_scenario(sid)
print("Scenario:", sid)
print("Agents:", scenario.agent_ids)
print("Common frames:", scenario.frame_indices[:10], "...")

frame_idx = scenario.frame_indices[0]
frames = ds.get_all_agent_frames(sid, frame_idx, load_lidar=False)
for aid, f in frames.items():
    print(f"  Agent {aid}: {len(f.gt_objects)} objects")

groups = OPV2VDataset.match_objects_across_agents(frames, tolerance_m=3.0)
print(f"Collective unique objects: {len(groups)}")
shared = [g for g in groups if len(g) > 1]
print(f"Shared (seen by >1 agent): {len(shared)}  |  Exclusive: {len(groups) - len(shared)}")

## 4. Run the full analysis

Start with a few scenarios (`max_scenarios`) for a quick run; set to `None` for all.

In [ ]:
summary = ac.run_analysis(
    data_root=DATA_ROOT,
    out_dir=Path("/kaggle/working/results/cooperation_coverage"),
    max_scenarios=None,       # e.g. 5 for a quick run; None = all
    match_tolerance_m=3.0,
)

print("=" * 60)
print("HEADLINE RESULT")
print("=" * 60)
print(f"Frames analyzed:            {summary.num_frames}")
print(f"Mean agents/frame:          {summary.mean_agents_per_frame:.2f}")
print(f"Mean objects (best single): {summary.mean_single_count:.2f}")
print(f"Mean objects (collective):  {summary.mean_collective_count:.2f}")
print(f"Gain vs best single agent:  {summary.mean_gain_vs_best_single:.2f}x")
print(f"Gain vs worst single agent: {summary.mean_gain_vs_worst_single:.2f}x")
print(f"Extra objects/frame:        +{summary.mean_extra_objects_vs_best_single:.1f}")

## 5. View the plot and top cooperation scenes

In [ ]:
from IPython.display import Image, display

plot_path = Path("/kaggle/working/results/cooperation_coverage/cooperation_coverage.png")
if plot_path.exists():
    display(Image(str(plot_path)))

print("\nTop cooperation scenes (biggest gain — demo candidates):")
for s in summary.top_cooperation_scenes[:5]:
    print(f"  {s['scenario_id']} frame {s['frame_idx']}: "
          f"{s['best_single_count']} -> {s['collective_count']} "
          f"({s['gain']}x, +{s['extra_objects']} objects)")

## 6. Kill Gate A read

- **Gain well above 1.0x** (e.g. collective sees clearly more than the best single agent) -> cooperation has strong headroom on this data -> **proceed** to Phase 2 (real detector) / Phase 3 (cooperative experiment).
- **Gain near 1.0x** -> the premise is weak on this data -> investigate (wrong scenes? agents too far apart? too few agents?) before building further.

Note: this is the ground-truth **ceiling** of cooperative benefit. The *achievable* gain with a real detector (which can miss objects) is measured in Phase 3.